# Multi-Turn Chatbot with Long-Term Memory

This notebook demonstrates building a multi-turn chatbot with:

- **Long-term memory** using Zep Cloud or Mem0
- **Tool calling** with web search (Tavily, Wikipedia) and more
- **Multi-turn conversation** that remembers context across sessions

## What You'll Learn

1. Configure Zep Cloud for persistent memory across sessions
2. Add search tools (Tavily, Wikipedia) and memory tools
3. Build a conversational chatbot that learns about you
4. View the conversation history saved by Zep

## Prerequisites

- **Zep Cloud API key**: Get one at [https://www.getzep.com](https://www.getzep.com)
- **NVIDIA API key**: Get one at [https://build.nvidia.com](https://build.nvidia.com)
- **Tavily API key** (optional): Get one at [https://tavily.com](https://tavily.com)
- **Plugins**: `uv pip install nvidia-nat-zep-cloud nvidia-nat-langchain`


## Step 1: Setup and Environment


In [1]:
import getpass
import os
import sys
from pathlib import Path

# For notebook async support
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# Load environment variables from .env file
load_dotenv()

# Memory thread ID - change this to start a fresh conversation
MEMORY_THREAD_ID = "chatbot_demo_thread"


async def clear_memory():
    """Clear the Zep memory thread to start fresh."""
    zep_key = os.environ.get("ZEP_API_KEY")
    if not zep_key:
        print("⏭️ Zep not configured")
        return

    try:
        from zep_cloud.client import AsyncZep

        client = AsyncZep(api_key=zep_key)
        await client.thread.delete(thread_id=MEMORY_THREAD_ID)
        print(f"✅ Cleared memory (thread: {MEMORY_THREAD_ID})")
    except Exception as e:
        if "not found" in str(e).lower():
            print(f"ℹ️ No previous memory found for thread: {MEMORY_THREAD_ID}")
        else:
            print(f"⚠️ Could not clear memory: {e}")


print("✅ Environment configured")
print(f"📝 Memory thread ID: {MEMORY_THREAD_ID}")


✅ Environment configured
📝 Memory thread ID: chatbot_demo_thread


In [ ]:
# Check for NVIDIA API key (required for LLM)
nvidia_api_key = os.environ.get("NVIDIA_API_KEY")

if nvidia_api_key:
    print("✅ NVIDIA_API_KEY loaded")
else:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA_API_KEY (get one at https://build.nvidia.com/): ")
    if nvidia_api_key:
        os.environ["NVIDIA_API_KEY"] = nvidia_api_key
        print("✅ NVIDIA_API_KEY set")
    else:
        print("⚠️ NVIDIA_API_KEY not set - LLM will not work")


✅ NVIDIA_API_KEY loaded


In [ ]:
# Check for Zep Cloud API key
zep_api_key = os.environ.get("ZEP_API_KEY")

if zep_api_key:
    print("✅ ZEP_API_KEY loaded")
else:
    zep_api_key = getpass.getpass(
        "Enter your ZEP_API_KEY (get one at https://www.getzep.com, or press Enter to skip): "
    )
    if zep_api_key:
        os.environ["ZEP_API_KEY"] = zep_api_key
        print("✅ ZEP_API_KEY set")
    else:
        print("⏭️ Skipping Zep - will try Mem0 instead")


✅ ZEP_API_KEY set


In [ ]:
# Check for Tavily API key (optional - for web search)
tavily_api_key = os.environ.get("TAVILY_API_KEY")

if tavily_api_key:
    print("✅ TAVILY_API_KEY loaded")
else:
    tavily_api_key = getpass.getpass(
        "Enter your TAVILY_API_KEY (get one at https://tavily.com, or press Enter to skip): ")
    if tavily_api_key:
        os.environ["TAVILY_API_KEY"] = tavily_api_key
        print("✅ TAVILY_API_KEY set")
    else:
        print("⏭️ Skipping Tavily - will use Wikipedia search instead")


NameError: name 'os' is not defined

## Step 2: Configure Long-Term Memory

We'll use **Zep Cloud** for long-term memory. Zep provides:

- Automatic conversation summarization
- Semantic memory search
- User preference extraction
- Multi-session persistence


In [5]:
memory_backend = None

# Try Zep Cloud first
if zep_api_key:
    try:
        from nat.plugins.zep_cloud.memory import ZepMemory

        memory_backend = ZepMemory(
            name="zep_memory",
        )
        print("✅ Zep Cloud Memory configured")
    except ImportError:
        print("⚠️ Zep plugin not installed. Run: uv pip install nvidia-nat-zep-cloud")

# Fall back to Mem0 if Zep is not available
if memory_backend is None:
    mem0_api_key = os.environ.get("MEM0_API_KEY")
    if not mem0_api_key:
        mem0_api_key = getpass.getpass("Enter your MEM0_API_KEY (or press Enter to skip memory): ")
        if mem0_api_key:
            os.environ["MEM0_API_KEY"] = mem0_api_key

    if mem0_api_key:
        try:
            from nat.plugins.mem0ai.memory import Mem0Memory

            memory_backend = Mem0Memory(
                name="mem0_memory",
            )
            print("✅ Mem0 Memory configured")
        except ImportError:
            print("⚠️ Mem0 plugin not installed. Run: uv pip install nvidia-nat-mem0ai")

if memory_backend is None:
    print("⚠️ No memory backend available - assistant will not remember between sessions")


✅ Zep Cloud Memory configured


## Step 3: Configure Tools

Our chatbot will have access to useful tools:

| Tool | Purpose |
|------|----------|
| **Tavily Search** | Search the web for current information |
| **Wikipedia Search** | Search Wikipedia for factual information |
| **Current Time** | Get the current date and time |
| **Add Memory** | Store information about the user |
| **Get Memory** | Retrieve stored information |


In [ ]:
from pydantic import SecretStr

from nat.tool.datetime_tools import CurrentTimeTool
from nat.tool.memory_tools.add_memory_tool import AddMemoryTool
from nat.tool.memory_tools.get_memory_tool import GetMemoryTool

# Initialize tools list
tools = []

# Add web search tool (Tavily or Wikipedia)
if tavily_api_key:
    try:
        from nat.plugins.langchain.tools.tavily_internet_search import TavilyInternetSearchTool

        tavily_tool = TavilyInternetSearchTool(
            name="web_search",
            max_results=3,
            api_key=SecretStr(tavily_api_key),  # Must be SecretStr, not plain string
        )
        tools.append(tavily_tool)
        print("✅ Added: web_search (Tavily)")
    except ImportError:
        print("⚠️ Tavily not available. Run: uv pip install nvidia-nat-langchain")
        tavily_api_key = None

# Fall back to Wikipedia if Tavily not available
if not tavily_api_key:
    try:
        from nat.plugins.langchain.tools.wikipedia_search import WikiSearchTool

        wiki_tool = WikiSearchTool(
            name="wiki_search",
            max_results=2,
        )
        tools.append(wiki_tool)
        print("✅ Added: wiki_search (Wikipedia)")
    except ImportError:
        print("⚠️ Wikipedia search not available. Run: uv pip install nvidia-nat-langchain")

# Add time tool
tools.append(CurrentTimeTool(name="current_time"))
print("✅ Added: current_time tool")

# Add memory tools if backend is available
if memory_backend:
    add_memory = AddMemoryTool(
        nat_memory=memory_backend,
        name="add_memory",
        description="Store important information about the user such as preferences, facts, names, or details.",
    )

    get_memory = GetMemoryTool(
        nat_memory=memory_backend,
        name="get_memory",
        description="Retrieve previously stored information about the user to personalize responses.",
    )

    tools.extend([add_memory, get_memory])
    print("✅ Added: add_memory, get_memory tools")

print(f"\n📦 Total tools configured: {len(tools)}")


✅ Added: web_search (Tavily)
✅ Added: current_time tool
✅ Added: add_memory, get_memory tools

📦 Total tools configured: 4


### (Optional) Clear Previous Memory

Run this cell to clear previous memory and start fresh. You can also change `MEMORY_THREAD_ID` in the setup cell above to use a different thread.


In [7]:
# Clear memory to start fresh
await clear_memory()


✅ Cleared memory (thread: chatbot_demo_thread)


## Step 4: Create the Multi-Turn Chatbot


In [8]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.3,  # Lower temperature for consistent ReAct format output
    max_tokens=1024,
    name="chatbot_llm",
)

# System instructions for the chatbot
system_instructions = """
You are a helpful multi-turn chatbot with long-term memory and web search capabilities.

Your capabilities:
1. **Search the web**: Use web_search or wiki_search to find current information
2. **Remember the user**: Use add_memory to store important information they share
3. **Recall context**: Use get_memory to recall what you know about the user
4. **Tell the time**: Use current_time when asked about time/date

IMPORTANT Guidelines:
- ALWAYS use add_memory immediately when the user tells you their name, preferences, or any personal detail
- When the user introduces themselves (e.g., "My name is X"), FIRST call add_memory to store their name, THEN respond
- Before answering questions about the user, ALWAYS call get_memory first
- When asked factual questions, search the web or Wikipedia
- Reference stored information naturally in conversation
- Keep responses concise but friendly and engaging
"""

# Create the agent
chatbot = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
    additional_instructions=system_instructions,
    # Retry parsing if LLM output doesn't match expected format
    retry_agent_response_parsing_errors=True,
    parse_agent_response_max_retries=5,
)

# Create workflow
workflow = NatWorkflow(entrypoint=chatbot)

print("✅ Multi-Turn Chatbot created!")
print(f"   - LLM: {llm.model_name}")
print(f"   - Tools: {[t.name for t in tools]}")
print(f"   - Memory: {memory_backend.computed_name if memory_backend else 'None'}")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Multi-Turn Chatbot created!
   - LLM: meta/llama-3.3-70b-instruct
   - Tools: ['web_search', 'current_time', 'add_memory', 'get_memory']
   - Memory: zep_memory


## Step 5: Multi-Turn Conversation

Let's have a conversation with the assistant. Watch how it:

1. Stores information you share
2. Recalls context in follow-up messages
3. Uses tools to help you


In [9]:
# Conversation Turn 1: Introduction
print("💬 You: Hi! My name is Alex and I'm a software engineer who loves hiking.")
print()

response = await workflow.prompt(
    "Hi! My name is Alex and I'm a software engineer who loves hiking.",
    conversation_id=MEMORY_THREAD_ID,
)
print(f"🤖 Chatbot:\n{response}")


💬 You: Hi! My name is Alex and I'm a software engineer who loves hiking.



None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


🤖 Chatbot:
Hi Alex! It's nice to meet you. I've taken note that you're a software engineer with a passion for hiking. What brings you here today?


In [10]:
# Conversation Turn 2: Web search
print("💬 You: What are the best hiking trails in Colorado?")
print()

response = await workflow.prompt(
    "What are the best hiking trails in Colorado?",
    conversation_id=MEMORY_THREAD_ID,
)
print(f"🤖 Chatbot:\n{response}")


💬 You: What are the best hiking trails in Colorado?

🤖 Chatbot:
Some of the best summer hikes in Colorado are Mount Bierstadt Trail, Emerald Lake Trail, Herman Gulch Trail, Lake Isabelle, Fountain Valley Trail, Castle/Meadow Trail, Paradise Cove, and Windy Saddle to Lookout Mountain.


In [11]:
# Conversation Turn 3: Share preferences
print("💬 You: I prefer morning hikes. And I always have coffee before I go - it's my favorite ritual.")
print()

response = await workflow.prompt(
    "I prefer morning hikes. And I always have coffee before I go - it's my favorite ritual.",
    conversation_id=MEMORY_THREAD_ID,
)
print(f"🤖 Chatbot:\n{response}")


💬 You: I prefer morning hikes. And I always have coffee before I go - it's my favorite ritual.



[AGENT] ReAct Agent wants to call tool None. In the ReAct Agent's configuration within the config file,there is no tool with that name: ['web_search', 'current_time', 'add_memory', 'get_memory']


🤖 Chatbot:
It's great that you enjoy morning hikes and have a favorite ritual of having coffee before you go. Do you have a favorite hiking spot or a particular type of coffee that you like to start your day with?


In [12]:
# Conversation Turn 4: Test memory recall
print("💬 You: What do you remember about me?")
print()

response = await workflow.prompt(
    "What do you remember about me?",
    conversation_id=MEMORY_THREAD_ID,
)
print(f"🤖 Chatbot:\n{response}")


💬 You: What do you remember about me?

🤖 Chatbot:
I remember that your name is Alex, you work as a software engineer, and you enjoy hiking, particularly in the mornings. You also have a favorite ritual of having coffee before going for a hike.


## Step 6: Save Configuration

Save the workflow configuration for CLI or production use.


In [13]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "multiturn_chatbot.yaml"
workflow.save_to_config_file(config_path)
print(f"📄 Configuration saved to: {config_path}")


📄 Configuration saved to: configs/multiturn_chatbot.yaml


## Step 7: View Zep Conversation History

Let's see what Zep has stored from our conversation. This demonstrates how Zep automatically extracts and organizes information.


In [14]:
# View conversation history stored in Zep
if zep_api_key and memory_backend:
    try:
        from zep_cloud.client import AsyncZep

        zep_client = AsyncZep(api_key=zep_api_key)

        print("📝 Zep Conversation Memory")
        print("=" * 50)
        print(f"Thread ID: {MEMORY_THREAD_ID}")

        try:
            context = await zep_client.thread.get_user_context(thread_id=MEMORY_THREAD_ID)
            if context.context:
                print("\n🧠 Extracted User Context:")
                print("-" * 40)
                print(context.context)
            else:
                print("\n(No context extracted yet - run more conversations!)")
        except Exception as e:
            print(f"Note: Could not retrieve context: {e}")

    except ImportError:
        print("⚠️ Zep client not available. Install with: uv pip install zep-cloud")
else:
    print("⏭️ Zep not configured - skipping conversation history view")


📝 Zep Conversation Memory
Thread ID: chatbot_demo_thread

🧠 Extracted User Context:
----------------------------------------


# This is the user summary
<USER_SUMMARY>
The user's name is Alex. A key personal lifestyle activity mentioned is hiking.

Alex works as a software engineer. Hiking is listed as a main pursuit.

Alex enjoys hiking and specifically prefers morning hikes. A favorite and recurring ritual for Alex is having coffee before hiking.
</USER_SUMMARY>


FACTS and ENTITIES, and EPISODES represent relevant context to the current conversation.

# These are the most relevant facts and their valid date ranges
# format: FACT (Date range: from - to)
# NOTE: Facts ending in "present" are currently valid (e.g., "Jane prefers her coffee with milk (2024-01-15 10:30:00 - present)" means Jane currently prefers coffee with milk)
#       Facts with a past end date used to be valid but are NOT CURRENTLY VALID (e.g., "Jane prefers her coffee with milk (2024-01-15 10:30:00 - 2024-06-20 14:

## Interactive Chat Mode

Run the cell below to start an interactive chat session with your chatbot.


In [15]:
# Clear memory before starting interactive chat
await clear_memory()

# Interactive chat mode
print("🤖 Multi-Turn Chatbot Ready!")
print(f"   Memory thread: {MEMORY_THREAD_ID}")
print("   Type your message and press Enter")
print("   Type 'quit' or 'exit' to end the conversation")
print("-" * 50)

while True:
    try:
        user_input = input("\n💬 You: ").strip()

        if not user_input:
            continue

        # Echo the user's message
        print(f"💬 You: {user_input}")

        if user_input.lower() in ["quit", "exit", "q"]:
            print("\n👋 Goodbye! Your memories have been saved to Zep.")
            break

        response = await workflow.prompt(user_input, conversation_id=MEMORY_THREAD_ID)
        print(f"\n🤖 Chatbot: {response}")

    except KeyboardInterrupt:
        print("\n\n👋 Goodbye!")
        break
    except Exception as e:
        print(f"\n❌ Error: {e}")


✅ Cleared memory (thread: chatbot_demo_thread)
🤖 Multi-Turn Chatbot Ready!
   Memory thread: chatbot_demo_thread
   Type your message and press Enter
   Type 'quit' or 'exit' to end the conversation
--------------------------------------------------
💬 You: My name is Sam and I am interested in NYC apartments. Can you make some suggestions?


[AGENT] ReAct Agent wants to call tool None. In the ReAct Agent's configuration within the config file,there is no tool with that name: ['web_search', 'current_time', 'add_memory', 'get_memory']



🤖 Chatbot: Hi Sam, I've found some websites that can help you find NYC apartments for rent. You can check out StreetEasy, RentHop, or Realtor.com to browse available apartments with your preferred amenities. Do you need more specific help or have any particular preferences for your apartment search?
💬 You: Can you give me some options on Jersey City apartments?

🤖 Chatbot: You can find Jersey City apartments on websites like Trulia, Apartments.com, and Newport Rentals. These websites offer a wide range of options, including studio, 1, 2, and 3 bedroom apartments. You can filter your search by neighborhood, schools, and local guides to find the perfect apartment for your needs.
💬 You: Great. Now what was my name?

🤖 Chatbot: Your name is Sam.
💬 You: And where did I ask you to look for apartments?

🤖 Chatbot: You didn't specify where you asked me to look for apartments, but I do know you're interested in finding one in NYC. Could you provide more context or clarify your question?
💬 You:

## Step 8: View Final Conversation History

Run this cell after your interactive chat session to see what Zep learned.


In [17]:
# View final conversation history from Zep
print("📝 Final Zep Conversation Memory")
print("=" * 50)
print(f"Thread ID: {MEMORY_THREAD_ID}")

if zep_api_key and memory_backend:
    try:
        from zep_cloud.client import AsyncZep

        zep_client = AsyncZep(api_key=zep_api_key)

        try:
            context = await zep_client.thread.get_user_context(thread_id=MEMORY_THREAD_ID)
            if context.context:
                print("\n🧠 Extracted User Context:")
                print("-" * 40)
                print(context.context)
            else:
                print("\n(No context extracted yet - have more conversations!)")
        except Exception as e:
            print(f"Note: Could not retrieve context: {e}")

    except ImportError:
        print("⚠️ Zep client not available. Install with: uv pip install zep-cloud")
else:
    print("⏭️ Zep not configured - skipping conversation history view")


📝 Final Zep Conversation Memory
Thread ID: chatbot_demo_thread

🧠 Extracted User Context:
----------------------------------------


# This is the user summary
<USER_SUMMARY>
The user's first name is Sam. The user has expressed interest in NYC apartments.

The user is interested in NYC apartments.
</USER_SUMMARY>


FACTS and ENTITIES represent relevant context to the current conversation.

# These are the most relevant facts and their valid date ranges
# format: FACT (Date range: from - to)
# NOTE: Facts ending in "present" are currently valid (e.g., "Jane prefers her coffee with milk (2024-01-15 10:30:00 - present)" means Jane currently prefers coffee with milk)
#       Facts with a past end date used to be valid but are NOT CURRENTLY VALID (e.g., "Jane prefers her coffee with milk (2024-01-15 10:30:00 - 2024-06-20 14:00:00)" means Jane no longer prefers coffee with milk)
<FACTS>
  - Sam is interested in NYC apartments. (2025-12-15 21:17:12 - present)
</FACTS>





## CLI Commands

```bash
# Run the chatbot
nat run --config_file configs/multiturn_chatbot.yaml --input "Hi, what do you know about me?"

# Start interactive console
nat serve --config_file configs/multiturn_chatbot.yaml
```

## Summary

In this notebook, you built a multi-turn chatbot with:

| Feature | Implementation |
|---------|----------------|
| **Long-term memory** | Zep Cloud (with Mem0 fallback) |
| **Web search** | Tavily or Wikipedia |
| **Tool calling** | CurrentTime, AddMemory, GetMemory |
| **Multi-turn conversation** | Workflow with context retention |
| **Memory inspection** | View extracted context from Zep |

### Key Classes Used

| Class | Purpose |
|-------|----------|
| `ZepMemory` / `Mem0Memory` | Long-term memory backend |
| `TavilyInternetSearchTool` | Web search via Tavily |
| `WikiSearchTool` | Wikipedia search |
| `AddMemoryTool` | Store user information |
| `GetMemoryTool` | Retrieve user information |
| `NatReActAgent` | Agent with reasoning and tool use |
| `NatWorkflow` | Workflow orchestration |

## Next Steps

- Add more tools (code execution, calculations, etc.)
- Deploy as an API using `nat serve`
- See [Memory documentation](../../../docs/source/build-workflows/memory.md) for advanced memory features
